<a href="https://colab.research.google.com/github/kanitabajrami/Computer-Graphics/blob/main/Campus_FAQ_Assistant_Week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Campus FAQ Assistant
ECS-609 Introduction to AI — Prototype and Evaluation
Goal: Help SEEU students find answers to common university questions. This prototype represents FAQ questions with TF-IDF, ranks them using cosine similarity, and rejects questions when the best match is below a threshold.

Development stage: Weeks 1–2 — data loading, first retrieval prototype, and preliminary evaluation. The dataset and manually written tests are small, so the scores are preliminary and should not be treated as proof of general performance.

## 1. Install and import libraries
Pandas is used to inspect the dataset. Scikit-learn provides TF-IDF and cosine similarity.

In [ ]:
%pip -q install pandas scikit-learn

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

What this does:
- pandas loads and manages your FAQ dataset.
- scikit-learn provides the tools for TF-IDF and cosine similarity.

## 2. Load the FAQ dataset
The CSV is stored in the project's GitHub repository. Check every answer and source URL against official SEEU information before describing it as verified.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/kanitabajrami/campus-faq-assistant/main/data/faq.csv"
faq_df = pd.read_csv(DATA_URL)

required_columns = {"faq_id", "question", "answer"}
missing_columns = required_columns - set(faq_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

faq_df = faq_df.dropna(subset=["faq_id", "question", "answer"]).copy()
faq_df["question"] = faq_df["question"].astype(str).str.strip()
faq_df["answer"] = faq_df["answer"].astype(str).str.strip()

print(f"Number of FAQ entries: {len(faq_df)}")
display(faq_df)

Number of FAQ entries: 5


,faq_id,question,answer,category,source_url,last_verified
0,FAQ001,Where can students find the academic calendar?,Students can find the academic calendar on the official SEEU website. Check the calendar for dates relevant to your ...,Calendar,https://www.seeu.edu.mk/en/current-students/calendar,2026-10-28
1,FAQ002,Where can students find the exam schedule?,"Check MySEEU for exam-session announcements, dates, and registration instructions. The academic calendar also provid...",Exams,https://www.seeu.edu.mk/en/registrar/studpar,2026-10-28
2,FAQ003,How do students register for courses?,SEEU's Semester Planning guidance states that students register for courses in MySEEU under Schedule Registration. C...,Registration,https://www.seeu.edu.mk/en/registrar/semesterplan,2026-10-28
3,FAQ004,Where can students find scholarship information?,SEEU publishes financial aid and scholarship information on its official website. Review the relevant scholarship sc...,Scholarships,https://www.seeu.edu.mk/en/future-students/financial-aid,2026-10-28
4,FAQ005,How can students request a transcript or certificate?,SEEU's academic information and Registry Office pages describe how students can obtain transcripts and certificates....,Documents,https://www.seeu.edu.mk/en/registrar/registaracademic,2026-10-28


## 3. Convert FAQ questions to TF-IDF vectors
TF-IDF converts text into numerical features. The vectorizer is fitted on the FAQ questions, and each new student question is transformed using the same fitted vectorizer.

In [ ]:
faq_questions = faq_df["question"].tolist()

vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
faq_vectors = vectorizer.fit_transform(faq_questions)

print("TF-IDF preparation completed.")
print("Number of FAQ questions:", len(faq_questions))
print("TF-IDF matrix shape:", faq_vectors.shape)

TF-IDF preparation completed.
Number of FAQ questions: 5
TF-IDF matrix shape: (5, 12)


## 4. Build the retrieval function
The system transforms a user question, compares it with every FAQ question, and selects the highest-scoring match. It returns the answer only when the best score reaches the threshold and is greater than zero. The threshold below is an initial setting, not an optimized value.

In [ ]:
DEFAULT_THRESHOLD = 0.20

def retrieve_faq(user_question, threshold=DEFAULT_THRESHOLD):
    """Return the best FAQ match, or reject a weak/empty query."""
    question = str(user_question).strip()
    if not question:
        return {"user_question": question, "similarity_score": 0.0,
                "predicted_faq_id": "OUT_OF_SCOPE", "matched_question": None,
                "answer": None, "accepted": False}

    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(question_vector, faq_vectors)[0]
    best_index = int(scores.argmax())
    best_score = float(scores[best_index])
    accepted = best_score >= threshold and best_score > 0

    if accepted:
        match = faq_df.iloc[best_index]
        predicted_id = match["faq_id"]
        matched_question = match["question"]
        answer = match["answer"]
    else:
        predicted_id, matched_question, answer = "OUT_OF_SCOPE", None, None

    return {"user_question": question, "similarity_score": round(best_score, 3),
            "predicted_faq_id": predicted_id, "matched_question": matched_question,
            "answer": answer, "accepted": accepted}

def ask_assistant(user_question, threshold=DEFAULT_THRESHOLD):
    result = retrieve_faq(user_question, threshold)
    print("Question:", result["user_question"])
    print("Similarity score:", result["similarity_score"])
    if result["accepted"]:
        print("Matched FAQ:", result["matched_question"])
        print("\nAnswer:")
        print(result["answer"])
    else:
        print("\nSorry, I couldn't find a sufficiently relevant answer.")
        print("Please check the official SEEU website or contact the relevant university office.")
    return result

## 6. Prepare the initial test set
Each question has an expected FAQ ID. OUT_OF_SCOPE means the current dataset does not contain an answer that adequately addresses the question. These manually written examples are useful for early testing, but the set is too small to establish general performance.

In [ ]:
initial_test_data = [
    {"question": "Where can I check the exam timetable?", "expected_faq_id": "FAQ002"},
    {"question": "How do I sign up for courses?", "expected_faq_id": "FAQ003"},
    {"question": "Where do I find scholarship details?", "expected_faq_id": "FAQ004"},
    {"question": "How can I get my transcript?", "expected_faq_id": "FAQ005"},
    {"question": "Where can I see the university calendar?", "expected_faq_id": "FAQ001"},
    {"question": "What is the best pizza in town?", "expected_faq_id": "OUT_OF_SCOPE"},
    {"question": "How do I fix my bicycle?", "expected_faq_id": "OUT_OF_SCOPE"},
    {"question": "When does the semester start?", "expected_faq_id": "OUT_OF_SCOPE"},
]
initial_test_df = pd.DataFrame(initial_test_data)
display(initial_test_df)

,question,expected_faq_id
0,Where can I check the exam timetable?,FAQ002
1,How do I sign up for courses?,FAQ003
2,Where do I find scholarship details?,FAQ004
3,How can I get my transcript?,FAQ005
4,Where can I see the university calendar?,FAQ001
5,What is the best pizza in town?,OUT_OF_SCOPE
6,How do I fix my bicycle?,OUT_OF_SCOPE
7,When does the semester start?,OUT_OF_SCOPE


## 7. Evaluate the initial test set
A prediction is correct if the system returns the expected FAQ ID or correctly rejects an out-of-scope question.

In [ ]:
def evaluate_questions(test_data, threshold=DEFAULT_THRESHOLD):
    results = pd.DataFrame(test_data).copy()
    predictions = results["question"].apply(lambda q: retrieve_faq(q, threshold))
    results["predicted_faq_id"] = predictions.apply(lambda r: r["predicted_faq_id"])
    results["similarity_score"] = predictions.apply(lambda r: r["similarity_score"])
    results["matched_question"] = predictions.apply(lambda r: r["matched_question"])
    results["correct"] = results["expected_faq_id"] == results["predicted_faq_id"]
    return results

def show_evaluation(results, title):
    print(title)
    print(f"Correct predictions: {int(results['correct'].sum())}/{len(results)}")
    print(f"Accuracy: {results['correct'].mean() * 100:.1f}%")
    display(results)

initial_results = evaluate_questions(initial_test_data)
show_evaluation(initial_results, "Initial test results")

Initial test results
Correct predictions: 8/8
Accuracy: 100.0%


,question,expected_faq_id,predicted_faq_id,similarity_score,matched_question,correct
0,Where can I check the exam timetable?,FAQ002,FAQ002,0.670,Where can students find the exam schedule?,True
1,How do I sign up for courses?,FAQ003,FAQ003,0.670,How do students register for courses?,True
2,Where do I find scholarship details?,FAQ004,FAQ004,0.670,Where can students find scholarship information?,True
3,How can I get my transcript?,FAQ005,FAQ005,0.557,How can students request a transcript or certificate?,True
4,Where can I see the university calendar?,FAQ001,FAQ001,0.670,Where can students find the academic calendar?,True
5,What is the best pizza in town?,OUT_OF_SCOPE,OUT_OF_SCOPE,0.000,None,True
6,How do I fix my bicycle?,OUT_OF_SCOPE,OUT_OF_SCOPE,0.000,None,True
7,When does the semester start?,OUT_OF_SCOPE,OUT_OF_SCOPE,0.000,None,True


## 8. Test more difficult questions
These examples use less direct wording and include questions that mention a related topic but may not be answerable by the current FAQ. Review the expected labels carefully: a question should only be labelled in-scope if the returned answer genuinely addresses it.

In [ ]:
challenging_test_data = [
    {"question": "Where do I look to find dates for exams?", "expected_faq_id": "FAQ002"},
    {"question": "I need to enroll in my subjects. Where do I go?", "expected_faq_id": "FAQ003"},
    {"question": "Are there any financial aid opportunities for students?", "expected_faq_id": "FAQ004"},
    {"question": "I need an official document showing my grades.", "expected_faq_id": "FAQ005"},
    {"question": "Where can I see important dates for the academic year?", "expected_faq_id": "FAQ001"},
    {"question": "How can I get a scholarship for studying abroad?", "expected_faq_id": "OUT_OF_SCOPE"},
    {"question": "What food is available at the university?", "expected_faq_id": "OUT_OF_SCOPE"},
    {"question": "How do I change my password?", "expected_faq_id": "OUT_OF_SCOPE"},
    {"question": "What are the graduation requirements?", "expected_faq_id": "OUT_OF_SCOPE"},
    {"question": "When is the registration deadline?", "expected_faq_id": "OUT_OF_SCOPE"},
]
challenging_results = evaluate_questions(challenging_test_data)
show_evaluation(challenging_results, "Challenging test results")

Challenging test results
Correct predictions: 5/10
Accuracy: 50.0%


,question,expected_faq_id,predicted_faq_id,similarity_score,matched_question,correct
0,Where do I look to find dates for exams?,FAQ002,OUT_OF_SCOPE,0.000,None,False
1,I need to enroll in my subjects. Where do I go?,FAQ003,OUT_OF_SCOPE,0.000,None,False
2,Are there any financial aid opportunities for students?,FAQ004,FAQ001,0.319,Where can students find the academic calendar?,False
3,I need an official document showing my grades.,FAQ005,OUT_OF_SCOPE,0.000,None,False
4,Where can I see important dates for the academic year?,FAQ001,FAQ001,0.670,Where can students find the academic calendar?,True
5,How can I get a scholarship for studying abroad?,OUT_OF_SCOPE,FAQ004,0.670,Where can students find scholarship information?,False
6,What food is available at the university?,OUT_OF_SCOPE,OUT_OF_SCOPE,0.000,None,True
7,How do I change my password?,OUT_OF_SCOPE,OUT_OF_SCOPE,0.000,None,True
8,What are the graduation requirements?,OUT_OF_SCOPE,OUT_OF_SCOPE,0.000,None,True
9,When is the registration deadline?,OUT_OF_SCOPE,OUT_OF_SCOPE,0.000,None,True


## 9. Inspect incorrect predictions
- False acceptance: an out-of-scope question is assigned an FAQ instead of being rejected.
- False rejection: an in-scope question is rejected as out of scope.
- Wrong FAQ: the system returns an FAQ, but it is not the expected one.
These cases help identify what needs improvement.

In [ ]:
all_results = pd.concat([
    initial_results.assign(test_set="Initial"),
    challenging_results.assign(test_set="Challenging")
], ignore_index=True)

errors = all_results.loc[~all_results["correct"]]
print(f"Questions across both sets: {len(all_results)}")
print(f"Incorrect predictions: {len(errors)}")
if errors.empty:
    print("No errors found in these test sets.")
else:
    display(errors[["test_set", "question", "expected_faq_id",
                    "predicted_faq_id", "similarity_score", "matched_question"]])

Questions across both sets: 18
Incorrect predictions: 5


,test_set,question,expected_faq_id,predicted_faq_id,similarity_score,matched_question
8,Challenging,Where do I look to find dates for exams?,FAQ002,OUT_OF_SCOPE,0.000,None
9,Challenging,I need to enroll in my subjects. Where do I go?,FAQ003,OUT_OF_SCOPE,0.000,None
10,Challenging,Are there any financial aid opportunities for students?,FAQ004,FAQ001,0.319,Where can students find the academic calendar?
11,Challenging,I need an official document showing my grades.,FAQ005,OUT_OF_SCOPE,0.000,None
13,Challenging,How can I get a scholarship for studying abroad?,OUT_OF_SCOPE,FAQ004,0.670,Where can students find scholarship information?


## 10. Evaluation summary
Keep the two test sets separate because they contain questions of different difficulty. These scores are based on manually prepared examples and should be treated as preliminary.

In [ ]:
summary_df = pd.DataFrame([
    {"Test set": "Initial", "Questions": len(initial_results),
     "Correct": int(initial_results["correct"].sum()),
     "Accuracy (%)": round(initial_results["correct"].mean() * 100, 1)},
    {"Test set": "Challenging", "Questions": len(challenging_results),
     "Correct": int(challenging_results["correct"].sum()),
     "Accuracy (%)": round(challenging_results["correct"].mean() * 100, 1)}
])
display(summary_df)

,Test set,Questions,Correct,Accuracy (%)
0,Initial,8,8,100.0
1,Challenging,10,5,50.0


## 11. Limitations and next steps
Current limitations
- The FAQ dataset is very small, so many legitimate student questions cannot be answered.
- TF-IDF relies on word overlap and may struggle with paraphrases or related terms that use different vocabulary.
- A high similarity score does not guarantee that the returned answer addresses the student's actual intent.
- The threshold is an initial choice and has not been systematically tuned.
- The tests are manually written and do not yet represent real student questions.
- FAQ answers and source URLs must be checked against official SEEU information.
Planned improvements
1. Expand the FAQ dataset using verified official sources.
2. Add a keyword-overlap baseline and compare it with TF-IDF.
3. Create a larger fixed test set with paraphrases, in-scope questions, and out-of-scope questions.
4. Review failure cases and tune the threshold using a separate validation set.
5. Add a simple user interface and clearer source links after retrieval quality improves.
Current conclusion
The first prototype retrieves answers for some straightforward questions. More difficult tests reveal weaknesses in paraphrase handling and rejection behaviour. The next stage is to improve verified FAQ coverage and evaluate the system systematically rather than relying on a small initial accuracy score.